# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HamzaMohamed25/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Setup | load the decision-point frame; build it from the warehouse only if it is not cached (features Feb-Apr 2026, label May 2026; June is never loaded)
import gc, json, os
import numpy as np, pandas as pd

FRAME = "work/outputs/frame_dp_2026-05-01.parquet"
os.makedirs("work/outputs", exist_ok=True)

def hf_token():
    try:
        from google.colab import userdata          # Colab: Secrets panel
        return userdata.get("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")          # local run: environment variable

if os.path.exists(FRAME):
    print("Using the cached frame:", FRAME)
else:
    # ML-04 v2 | Decision-point frame: features Feb-Apr 2026, label May 2026. June stays sealed (never loaded).
    import gc, json, os
    import numpy as np, pandas as pd

    HF = {"token": hf_token()}
    HF_BASE = "hf://datasets/FlyRank/internship-warehouse"
    FEAT_MONTHS, LABEL_MONTH = ["2026-02", "2026-03", "2026-04"], "2026-05"
    MIN_IMPR_WINDOW, MIN_CLICKS_APR, DROP_RATIO = 1000, 10, 0.80      # contract thresholds: frozen before any model
    KEYS = ["client_hash_id", "content_hash_id"]
    COLS = ["report_date", *KEYS, "gsc_clicks", "gsc_impressions", "gsc_avg_position"]
    assert LABEL_MONTH != "2026-06" and "2026-06" not in FEAT_MONTHS, "June is the sealed test month"

    def month_agg(m):
        """One partition -> one row per (client, content). Loads only needed columns; frees RAM right after."""
        path = f"{HF_BASE}/fact_content_daily_performance/month={m}/"
        try:
            d = pd.read_parquet(path, columns=COLS, storage_options=HF)
        except Exception:
            from huggingface_hub import HfFileSystem
            import pyarrow.parquet as pq
            fs = HfFileSystem(token=HF["token"])
            f = fs.ls(f"datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={m}", detail=False)[0]
            print("Actual schema:", pq.read_schema(f, filesystem=fs).names); raise
        dates = pd.to_datetime(d["report_date"])
        win = (dates.min().date(), dates.max().date(), len(d))
        impr = d["gsc_impressions"].fillna(0)
        pos = d["gsc_avg_position"].where(d["gsc_avg_position"] > 0)            # 0/negative = no data, not rank 0
        d = d.assign(clicks=d["gsc_clicks"].fillna(0), impr=impr,
                     pos_x_impr=(pos * impr).fillna(0), impr_pos=impr.where(pos.notna(), 0),
                     active=(impr > 0).astype("int16"))
        g = d.groupby(KEYS, observed=True).agg(clicks=("clicks", "sum"), impr=("impr", "sum"),
                                               pos_x_impr=("pos_x_impr", "sum"), impr_pos=("impr_pos", "sum"),
                                               active_days=("active", "sum")).reset_index()
        del d; gc.collect()
        return g, win

    # 1) Aggregate each month separately (daily grain -> content grain), check windows land inside their month
    frames, windows = {}, {}
    for m in FEAT_MONTHS + [LABEL_MONTH]:
        frames[m], windows[m] = month_agg(m)
        assert str(windows[m][0]).startswith(m) and str(windows[m][1]).startswith(m), f"{m} partition has dates outside its month"
        print(f"{m}: daily rows={windows[m][2]:,} | {windows[m][0]} -> {windows[m][1]} | contents={len(frames[m]):,}")

    # 2) Feature frame from Feb-Apr ONLY (nine features, all knowable on 2026-05-01)
    f = None
    for m in FEAT_MONTHS:
        part = frames[m].rename(columns={c: f"{c}_{m}" for c in ["clicks", "impr", "pos_x_impr", "impr_pos", "active_days"]})
        f = part if f is None else f.merge(part, on=KEYS, how="outer")
    num = [c for c in f.columns if c not in KEYS]
    f[num] = f[num].fillna(0)                                             # absent row in a month = no search rows that month
    S = lambda p: sum(f[f"{p}_{m}"] for m in FEAT_MONTHS)
    f["impressions_total"], f["clicks_total"] = S("impr"), S("clicks")
    f["impressions_apr"], f["clicks_apr"], f["clicks_feb"] = f["impr_2026-04"], f["clicks_2026-04"], f["clicks_2026-02"]
    f["momentum_apr_vs_feb"] = np.log1p(f["clicks_apr"]) - np.log1p(f["clicks_feb"])
    f["ctr"] = np.where(f["impressions_total"] > 0, f["clicks_total"] / f["impressions_total"], np.nan)
    f["weighted_position"] = np.where(S("impr_pos") > 0, S("pos_x_impr") / S("impr_pos").replace(0, np.nan), np.nan)  # sum(pos*impr)/sum(impr), not a mean of daily means
    f["active_days"] = S("active_days")
    FEATURES = ["impressions_total", "clicks_total", "impressions_apr", "clicks_apr", "clicks_feb",
                "momentum_apr_vs_feb", "ctr", "weighted_position", "active_days"]
    f = f[KEYS + FEATURES]

    # 3) Label from May ONLY: decline = May clicks < 80% of April clicks. Label ingredients are never features.
    may = frames[LABEL_MONTH][KEYS + ["clicks"]].rename(columns={"clicks": "clicks_may"})
    n_before_eligibility = len(f)
    f = f[(f["impressions_total"] >= MIN_IMPR_WINDOW) & (f["clicks_apr"] >= MIN_CLICKS_APR)]   # eligibility, decided at the decision point
    f = f.merge(may, on=KEYS, how="left")
    missing_in_may = int(f["clicks_may"].isna().sum())
    f["clicks_may"] = f["clicks_may"].fillna(0)                           # not seen in May -> 0 clicks -> counts as decline (stated in contract)
    f["y_decline_may"] = (f["clicks_may"] < DROP_RATIO * f["clicks_apr"]).astype(int)
    LABEL_INGREDIENTS = ["clicks_may"]                                    # kept ONLY to build/audit the label
    frame = f.reset_index(drop=True)

    # 4) Contract checks (queries, not claims)
    assert not frame.duplicated(KEYS).any(), "grain broken: (client, content) must be unique"
    assert not set(LABEL_INGREDIENTS) & set(FEATURES), "label ingredient leaked into features"
    assert set(KEYS).isdisjoint(FEATURES), "IDs are grouping keys, never features"
    share = frame["client_hash_id"].value_counts(normalize=True)
    base_rate = float(frame["y_decline_may"].mean())
    print(f"\nBefore eligibility: {n_before_eligibility:,} | Eligible rows: {len(frame):,} | Clients: {frame['client_hash_id'].nunique()}")
    print(f"Base rate (decline in May): {base_rate:.3f} | Not seen in May (counted as decline): {missing_in_may:,}")
    print(f"Largest client share of rows: {share.iloc[0]:.1%} | Top-5 clients: {share.head(5).sum():.1%}")
    print("Per-client base rate spread (min / median / max, clients with >=30 rows):")
    pc = frame.groupby("client_hash_id")["y_decline_may"].agg(["mean", "size"]); pc = pc[pc["size"] >= 30]["mean"]
    print(f"  {pc.min():.2f} / {pc.median():.2f} / {pc.max():.2f}  over {len(pc)} clients")
    print("Missing in features (share):"); print(frame[FEATURES].isna().mean().round(3).to_string())

    # 5) Receipt (commit this JSON), cache the frame locally (do NOT commit the parquet)
    os.makedirs("work/outputs", exist_ok=True)
    frame.to_parquet("work/outputs/frame_dp_2026-05-01.parquet", index=False)
    json.dump({"decision_point": "2026-05-01", "feature_months": FEAT_MONTHS, "label_month": LABEL_MONTH,
               "min_impr_window": MIN_IMPR_WINDOW, "min_clicks_apr": MIN_CLICKS_APR, "drop_ratio": DROP_RATIO,
               "rows_before_eligibility": int(n_before_eligibility), "rows": int(len(frame)),
               "clients": int(frame["client_hash_id"].nunique()), "base_rate": round(base_rate, 4),
               "not_seen_in_may": missing_in_may, "features": FEATURES, "label": "y_decline_may",
               "group_key": "client_hash_id", "sealed_month": "2026-06", "release": "v20260703"},
              open("work/outputs/data_contract_v2.json", "w"), indent=2)
    print("\nSaved: work/outputs/data_contract_v2.json (commit) + frame parquet (cache only)")

2026-02: daily rows=7,355,108 | 2026-02-01 -> 2026-02-28 | contents=321,546
2026-03: daily rows=9,841,378 | 2026-03-01 -> 2026-03-31 | contents=331,437
2026-04: daily rows=10,424,730 | 2026-04-01 -> 2026-04-30 | contents=362,172
2026-05: daily rows=11,687,376 | 2026-05-01 -> 2026-05-31 | contents=389,153

Before eligibility: 380,147 | Eligible rows: 16,513 | Clients: 36
Base rate (decline in May): 0.416 | Not seen in May (counted as decline): 0
Largest client share of rows: 26.1% | Top-5 clients: 71.9%
Per-client base rate spread (min / median / max, clients with >=30 rows):
  0.03 / 0.38 / 0.75  over 17 clients
Missing in features (share):
impressions_total      0.0
clicks_total           0.0
impressions_apr        0.0
clicks_apr             0.0
clicks_feb             0.0
momentum_apr_vs_feb    0.0
ctr                    0.0
weighted_position      0.0
active_days            0.0

Saved: work/outputs/data_contract_v2.json (commit) + frame parquet (cache only)


## 1. My rule and its reason codes

**The rule in plain words.** Flag a page when its April clicks are at least **2x its Feb-Mar average** (an April surge), but only if the page already had **at least 10 clicks in both February and March**. Rank the flagged pages by *surge strength x April impressions percentile* (bigger pages first). The idea is a hypothesis: a page that surged in April may fall back in May, so a person should **verify the spike** first (tracking? campaign? seasonal?) before anyone edits the page.

**Reason code (one):** `april_click_surge_vs_prior`. **Action label:** `verify_spike_then_review`. Every other page gets `no_flag` / `monitor` and score 0.

**Data check before anything else.** 833 pages from a single client have an *identical* Feb-Apr feature vector (same clicks, impressions, position and active days) and 99.6% of them declined in May. Identical rows are not independent evidence, so they are **excluded** from the analysis frame and from the queue. For transparency the pooled numbers that include them are printed (precision@50 = 0.74) but they are **not** the headline. The analysis frame has 15,680 rows and an observed base rate of 38.5%.

**Signal checks** (decline = May clicks < 80% of April clicks; n is printed in every table below):

- **Signal A - consecutive monthly click falls: OPPOSITE (weak).** Pages with two falls declined 33.9% (n=1,527) versus 39.3% for pages with no fall (n=7,076). My first idea, "it keeps falling, so refresh it", pointed the wrong way, so I dropped it.
- **Signal B - CTR versus position-peers (the logic behind the CTR-fix flag): FALSE.** The four quartiles sit at 39.6%, 38.5%, 38.7% and 37.2% (n about 3,920 each), a spread of 2.4 points. There is no usable pattern, so the rule has no CTR condition.
- **Signal C - April clicks versus the Feb-Mar average: CONFIRMED, but modest.** Pages above 2x declined 44.2% (n=3,798) versus about 36% for flat or falling pages. The spread is 8.4 points. This is the signal the rule uses.

The two official signal checks are **B (flag-linked)** and **C (behind the rule)**. A is kept as a clearly explained negative result.

**Disclosures.** The rule direction was chosen *after* the signal check. The history guard (10 clicks in Feb and Mar) was added *after* reviewing the weak picks of a first version without it. The duplicate block was excluded *after* diagnostics. All three are recorded in the metrics JSON. The thresholds (2x, 10 clicks, 20 copies) were fixed before the final run and are frozen now.

In [2]:
# 1) Load the frame, check the data, run the three signal checks
SEED = 42
df = pd.read_parquet(FRAME).reset_index(drop=True)
KEYS, LABEL = ["client_hash_id", "content_hash_id"], "y_decline_may"
MIN_RATIO, HISTORY_FLOOR, BLOCK_COPIES = 2.0, 10, 20

# Rule inputs: Feb-Apr only. clicks_mar is exact: the frame stores the Feb+Mar+Apr total plus Feb and Apr separately.
df["clicks_mar"] = (df["clicks_total"] - df["clicks_apr"] - df["clicks_feb"]).clip(lower=0)
prior = (df["clicks_feb"] + df["clicks_mar"]) / 2                       # Feb-Mar average clicks
df["ratio_apr"] = df["clicks_apr"] / prior.where(prior > 0)             # NaN when there were no Feb-Mar clicks (cannot judge a surge)
df["_tb"] = pd.util.hash_pandas_object(df["content_hash_id"].astype(str) + f"|{SEED}", index=False).to_numpy()   # tie policy: seeded hash
RULE_INPUTS = ["clicks_feb", "clicks_mar", "clicks_apr", "impressions_apr", "weighted_position", "active_days"]
assert not set(RULE_INPUTS) & {LABEL, "clicks_may"}, "label or its ingredient is in the rule inputs"

def run_rule(f, guard):
    """FLAG: April clicks >= 2x the Feb-Mar average (+ history guard). SCORE = 100 x surge strength (log2 ratio, capped 8x) x April impressions percentile."""
    flag = f["ratio_apr"] >= MIN_RATIO
    if guard: flag = flag & (f["clicks_feb"] >= HISTORY_FLOOR) & (f["clicks_mar"] >= HISTORY_FLOOR)
    strength = (np.log2(f["ratio_apr"].clip(lower=1)) / 3).clip(0, 1)
    score = pd.Series(np.where(flag, 100 * strength * f["impressions_apr"].rank(pct=True), 0.0), index=f.index).round(4)
    order = np.lexsort((f["_tb"].to_numpy(), -score.to_numpy()))        # score desc, then hash asc
    y = f[LABEL].to_numpy()[order]
    return flag, score, order, {f"precision_at_{k}": round(float(y[:k].mean()), 4) for k in (10, 50, 100)}

# Data-quality check: a block of pages with an IDENTICAL feature vector is not independent evidence
df["same_vector_copies"] = df.groupby(RULE_INPUTS, dropna=False)["content_hash_id"].transform("size")
df["templated_block"] = df["same_vector_copies"] >= BLOCK_COPIES
blk = df[df["templated_block"]]
_, _, _, pooled_prec = run_rule(df, True)                               # kept ONLY for disclosure; NOT the headline
print(f"Rows: {len(df):,} | Clients: {df['client_hash_id'].nunique()} | Base rate (all rows): {df[LABEL].mean():.3f}")
print(f"Templated block (same Feb-Apr vector, >= {BLOCK_COPIES} copies): {len(blk):,} rows in {blk.groupby(RULE_INPUTS, dropna=False).ngroups} group(s), "
      f"{blk['client_hash_id'].nunique()} client(s), decline rate {blk[LABEL].mean():.3f}")
print(f"Pooled precision@K INCLUDING the block (disclosure only, not the headline): {pooled_prec}\n")
n_block = len(blk); block_rate = float(blk[LABEL].mean()) if n_block else None
df = df[~df["templated_block"]].reset_index(drop=True)                  # analysis frame from here on
BASE = float(df[LABEL].mean())
print(f"Analysis frame (block excluded): {len(df):,} rows | base rate {BASE:.3f}\n")

# Signal checks on the analysis frame: bucket table + n + verdict (the verdict printed is a suggestion; the final words are in the markdown above)
def signal_table(bucket, name, hypothesis):
    """hypothesis: +1 = higher bucket should mean MORE decline, -1 = LESS decline."""
    t = df.assign(_b=bucket).dropna(subset=["_b"]).groupby("_b", observed=True)[LABEL].agg(n="size", decline_rate="mean")
    t["vs_base"] = (t["decline_rate"] - BASE).round(3); t["decline_rate"] = t["decline_rate"].round(3)
    r = t["decline_rate"].to_numpy(); steps = np.sign(np.diff(r)) * hypothesis
    if t["n"].min() < 30:         v = "MIXED (a bucket has n<30, too thin to call)"
    elif r.max() - r.min() < .03: v = "FALSE"
    elif (steps >= 0).all():      v = "CONFIRMED"
    elif (steps <= 0).all():      v = "OPPOSITE"
    else:                         v = "MIXED"
    print(f"=== {name} | hypothesis: {'higher bucket -> more decline' if hypothesis > 0 else 'higher bucket -> less decline'}")
    print(t.to_string()); print(f"Total n = {int(t['n'].sum()):,} | base rate = {BASE:.3f} | spread = {r.max() - r.min():.3f} | suggested verdict: {v}\n")
    return {"signal": name, "verdict": v, "table": t.reset_index().astype({"_b": str}).to_dict("records")}

n_falls = (df["clicks_mar"] < df["clicks_feb"]).astype(int) + (df["clicks_apr"] < df["clicks_mar"]).astype(int)
sigA = signal_table(n_falls.map({0: "0 falls", 1: "1 fall", 2: "2 falls"}).astype(pd.CategoricalDtype(["0 falls", "1 fall", "2 falls"], ordered=True)),
                    "A. consecutive monthly click falls", +1)
band = pd.cut(df["weighted_position"], [0, 3, 10, 20, np.inf], labels=["1-3", "4-10", "11-20", "21+"])   # flag-linked: CTR-vs-position logic
peer = df.groupby(band, observed=True)["ctr"].transform("median")
ratio = (df["ctr"] / peer.where(peer > 0)).replace([np.inf, -np.inf], np.nan)
qB = pd.qcut(ratio.dropna().rank(method="first"), 4, labels=["Q1 lowest CTR vs peers", "Q2", "Q3", "Q4 highest"])
sigB = signal_table(qB.reindex(df.index), "B. CTR vs position-peers (quartiles)", -1)
cat = pd.cut(df["ratio_apr"], [0, 0.5, 0.8, 1.25, 2, np.inf],
             labels=["a) Apr < 0.5x prior", "b) 0.5-0.8x", "c) 0.8-1.25x (flat)", "d) 1.25-2x", "e) > 2x (surge)"], include_lowest=True)
sigC = signal_table(cat, "C. April clicks vs Feb-Mar average", +1)
print(f"Rows with no Feb-Mar clicks (ratio undefined, never flagged): {int(df['ratio_apr'].isna().sum()):,}")

Rows: 16,513 | Clients: 36 | Base rate (all rows): 0.416
Templated block (same Feb-Apr vector, >= 20 copies): 833 rows in 1 group(s), 1 client(s), decline rate 0.996
Pooled precision@K INCLUDING the block (disclosure only, not the headline): {'precision_at_10': 0.4, 'precision_at_50': 0.74, 'precision_at_100': 0.87}

Analysis frame (block excluded): 15,680 rows | base rate 0.385

=== A. consecutive monthly click falls | hypothesis: higher bucket -> more decline
            n  decline_rate  vs_base
_b                                  
0 falls  7076         0.393    0.008
1 fall   7077         0.388    0.002
2 falls  1527         0.339   -0.046
Total n = 15,680 | base rate = 0.385 | spread = 0.054 | suggested verdict: OPPOSITE

=== B. CTR vs position-peers (quartiles) | hypothesis: higher bucket -> less decline
                           n  decline_rate  vs_base
_b                                                 
Q1 lowest CTR vs peers  3920         0.396    0.011
Q2                     

## 2. Build the ranked queue (writes the CSV)

**Score** = 100 x min(log2(April / Feb-Mar average) / 3, 1) x April-impressions percentile, only for flagged pages. **Ties** are broken by a seeded hash of the content id, so the order repeats on every run. The label (May) is used only *after* ranking, to evaluate.

**Observed results** on 15,680 pages (base rate 38.5%, which is also what random picking gives):

- Version without the history guard: 4,058 pages flagged, 44.6% declined; precision@10 / @50 / @100 = 0.30 / 0.38 / 0.40.
- **Final rule (with the guard): 435 pages flagged (2.8%), 42.1% declined; precision@10 / @50 / @100 = 0.50 / 0.42 / 0.37.**

**Honest reading.** The rule shows a small, directional lift in the whole flagged group (about 3.6 points over the base rate), but at the top of the queue it is **not clearly different from random**: with 435 pages the margin of chance is about +/-4.7 points, and a 50-page list can move by roughly +/-13 points by chance alone. That is fine for a baseline. It is transparent and traceable, and it sets the bar the Week-5 model has to beat on the same rows, folds, metric and K. This is decision-support only: no claim that fixing a flagged page would help.

In [3]:
# ONE reason code: april_click_surge_vs_prior | action: verify_spike_then_review (rest: no_flag / monitor, score 0).
flag0, _, _, prec0 = run_rule(df, False)
flag, df["baseline_score"], order, prec = run_rule(df, True)
df["reason_code"] = np.where(flag, "april_click_surge_vs_prior", "no_flag")
df["action"] = np.where(flag, "verify_spike_then_review", "monitor")
q_ = df.iloc[order].reset_index(drop=True)
q_["rank"] = np.arange(1, len(q_) + 1)

# Write the queue (stays out of git; the label is NOT in the queue file)
QCOLS = ["rank", *KEYS, "baseline_score", "reason_code", "action", "clicks_feb", "clicks_mar", "clicks_apr", "ratio_apr", "impressions_apr", "weighted_position"]
q_[QCOLS].to_csv("work/outputs/baseline_action_score.csv", index=False)

# Evaluation (label used ONLY here)
cut = q_.loc[49, "baseline_score"]; tied_at_cut = int((q_["baseline_score"] == cut).sum())
top_client_share = float(df.loc[flag, "client_hash_id"].value_counts(normalize=True).iloc[0]) if flag.any() else 0.0
print(f"v2 (no history guard): flagged {int(flag0.sum()):,} | decline rate among flagged {df.loc[flag0, LABEL].mean():.3f} | {prec0}")
print(f"v3 (history guard)   : flagged {int(flag.sum()):,} of {len(df):,} ({flag.mean():.1%}) | decline rate among flagged {df.loc[flag, LABEL].mean():.3f} | {prec}")
print(f"Random expectation = base rate = {BASE:.4f}")
print(f"Rows tied exactly at the top-50 cutoff ({cut:.4f}): {tied_at_cut} | clients inside top-50: {q_.head(50)['client_hash_id'].nunique()} | largest client share among flagged: {top_client_share:.1%}")
print("Saved: work/outputs/baseline_action_score.csv (do NOT commit)")

v2 (no history guard): flagged 4,058 | decline rate among flagged 0.446 | {'precision_at_10': 0.3, 'precision_at_50': 0.38, 'precision_at_100': 0.4}
v3 (history guard)   : flagged 435 of 15,680 (2.8%) | decline rate among flagged 0.421 | {'precision_at_10': 0.5, 'precision_at_50': 0.42, 'precision_at_100': 0.37}
Random expectation = base rate = 0.3854
Rows tied exactly at the top-50 cutoff (50.5836): 1 | clients inside top-50: 8 | largest client share among flagged: 22.5%
Saved: work/outputs/baseline_action_score.csv (do NOT commit)


## 3. Top-20 review

*Each row below shows the action, the reason code, a confidence note, why the page is there, and what would make it wrong.* The confidence note is **low** when the jump is extreme (more than 7x) or April clicks are below 30, otherwise **medium**. It is never **high**, because this is a weak baseline.

**What the review shows.** Every top-20 page is a spike, so "verify the spike" is the right first action. The queue never says "rewrite". The recurring failure mode is a *durable climb*: in the first 10 rows, ranks 1, 2, 3, 8 and 10 did **not** decline in May (for example rank 1 went 97, 119, 879 clicks and then 1,460 in May). A reviewer who checks whether the climb is a campaign, a seasonal topic or real growth would catch these before touching the page.

In [4]:
def wrong_if(r):
    w = []
    if r.ratio_apr > 7: w.append("extreme jump: check for a tracking or campaign artifact first")
    if r.active_days < 60: w.append("few active days: a new page still ramping up, not a spike to fade")
    w.append("seasonal topic, or a durable climb that keeps growing in May")
    return "; ".join(w[:3])
def confidence(r):   # never "high": this is a weak baseline
    return "low (extreme jump or small April base)" if (r.ratio_apr > 7 or r.clicks_apr < 30) else "medium"
top = q_.head(20).copy()
for r in top.itertuples():
    why = f"clicks Feb {r.clicks_feb:.0f}, Mar {r.clicks_mar:.0f} -> Apr {r.clicks_apr:.0f} ({r.ratio_apr:.1f}x prior), {r.impressions_apr:,.0f} impr in Apr, pos {r.weighted_position:.1f}"
    print(f"#{r.rank} [{r.action} | {r.reason_code}] confidence: {confidence(r)} | {r.content_hash_id[:12]}\n   why: {why}\n   wrong if: {wrong_if(r)}")

#1 [verify_spike_then_review | april_click_surge_vs_prior] confidence: low (extreme jump or small April base) | content_6121
   why: clicks Feb 97, Mar 119 -> Apr 879 (8.1x prior), 135,160 impr in Apr, pos 2.1
   wrong if: extreme jump: check for a tracking or campaign artifact first; seasonal topic, or a durable climb that keeps growing in May
#2 [verify_spike_then_review | april_click_surge_vs_prior] confidence: low (extreme jump or small April base) | content_86c9
   why: clicks Feb 10, Mar 19 -> Apr 144 (9.9x prior), 50,761 impr in Apr, pos 21.1
   wrong if: extreme jump: check for a tracking or campaign artifact first; seasonal topic, or a durable climb that keeps growing in May
#3 [verify_spike_then_review | april_click_surge_vs_prior] confidence: medium | content_2bed
   why: clicks Feb 32, Mar 248 -> Apr 924 (6.6x prior), 51,502 impr in Apr, pos 3.9
   wrong if: seasonal topic, or a durable climb that keeps growing in May
#4 [verify_spike_then_review | april_click_surge_vs_prio

## 4. Weak picks + leakage check

**Weak picks.** 29 of the top 50 did **not** decline in May. The pattern is the same as in the review above: the page kept climbing or held its April level (durable growth, a campaign still running, or a seasonal topic). The rule cannot tell a spike that fades from a rise that stays. A longer history, a seasonality flag, or a model that combines several signals are the obvious next tests, but this baseline does not claim any of them.

**Leakage check (all confirmed in code).**

- Rule inputs are **Feb-Apr 2026 only** (`clicks_feb`, `clicks_mar`, `clicks_apr`, `impressions_apr`, `weighted_position`, `active_days`). `clicks_mar` is derived exactly from the stored totals.
- The label and its ingredient `clicks_may` are used **only to evaluate**, never to score. The queue CSV does not contain them.
- **No product flags** (such as a ready-made trend or decay flag) and **no future window** are used. June 2026 is never loaded.
- IDs are used only for joining and for the tie-break hash, never as features.
- The duplicate block, the guard and the rule direction are disclosed above and in the metrics JSON.

Claims in this notebook are **observed** and **measured** on one decision point (1 May 2026), they are **directional** at best, and the queue is **decision-support**, not a causal claim.

In [5]:
weak = q_.head(50).query(f"{LABEL} == 0")[["rank", "clicks_feb", "clicks_mar", "clicks_apr", "clicks_may", "impressions_apr"]]
print(f"Weak picks: {len(weak)} of the top 50 did NOT decline in May (clicks_may shown for evaluation only). First 8:")
print(weak.head(8).to_string(index=False))
assert set(RULE_INPUTS).isdisjoint({LABEL, "clicks_may"}) and (q_.head(1)["reason_code"] == "april_click_surge_vs_prior").all()
print("\nLeakage check passed: rule inputs are Feb-Apr only; the label and clicks_may are used for evaluation, never for scoring.")

# Receipt (commit this JSON)
json.dump({"rule": "April clicks >= 2x Feb-Mar average, with >=10 clicks in Feb and in Mar; score = 100 * min(log2(ratio)/3, 1) * impressions_pctile_apr",
           "min_ratio": MIN_RATIO, "history_floor": HISTORY_FLOOR, "tie_break": f"hash(content_hash_id|{SEED})",
           "disclosures": ["direction chosen after signal check", "history guard added after v2 weak picks", "templated block excluded after diagnostics"],
           "templated_block": {"min_copies": BLOCK_COPIES, "rows_excluded": int(n_block), "decline_rate": None if block_rate is None else round(block_rate, 4)},
           "pooled_including_block_not_headline": pooled_prec,
           "rows_analysis_frame": int(len(df)), "base_rate": round(BASE, 4), "n_flagged": int(flag.sum()),
           "flagged_decline_rate": round(float(df.loc[flag, LABEL].mean()), 4), **prec,
           "v2_no_guard": {"n_flagged": int(flag0.sum()), "flagged_decline_rate": round(float(df.loc[flag0, LABEL].mean()), 4), **prec0},
           "tied_rows_at_top50_cutoff": tied_at_cut, "largest_client_share_among_flagged": round(top_client_share, 4),
           "signals": [sigA, sigB, sigC]},
          open("work/outputs/baseline_metrics.json", "w"), indent=2, default=str)
print("Saved: work/outputs/baseline_metrics.json (commit this one)")


Weak picks: 29 of the top 50 did NOT decline in May (clicks_may shown for evaluation only). First 8:
 rank  clicks_feb  clicks_mar  clicks_apr  clicks_may  impressions_apr
    1        97.0       119.0       879.0        1460         135160.0
    2        10.0        19.0       144.0         168          50761.0
    3        32.0       248.0       924.0        1136          51502.0
    8        10.0        13.0        60.0         107          27332.0
   10        14.0        15.0        73.0         143          26054.0
   11        10.0        28.0        92.0          84          31018.0
   12        24.0       221.0       550.0         590          62058.0
   13        14.0        17.0        78.0          78          22556.0

Leakage check passed: rule inputs are Feb-Apr only; the label and clicks_may are used for evaluation, never for scoring.
Saved: work/outputs/baseline_metrics.json (commit this one)


## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.